# Exact grid posterior (no MCMC)

Independent counterpart of the assimilation step in `clogging_prognostics.assimilation`. In every
refinement iteration the calibration posterior is a **1-D** function of the HSIC-active parameter
(the other inputs are held at their training mean), so it can be computed exactly on a grid:

1. same refine loop as the pipeline: DoE -> HSIC-active variable -> 15-member KLE-GP calibration
   ensemble (3 trends x 5 kernels) on that variable;
2. the likelihood is evaluated on a grid over the active marginal's support, aggregating the 15
   members with a **fixed scrambled-Sobol design** of `NB_WEIGHTS` simplex weights and a proper
   log-sum-exp (`log mean_w exp(ll_w)`), exactly as in the updated `mcmc_integration.py`;
3. posterior $\propto$ likelihood on the (uniform) support -- no sampling noise, no chains;
4. the posterior replaces the active marginal (as a `ot.UserDefined` on the grid), the DoE is
   regenerated through the base surrogate, and the loop repeats until the exact KL(posterior || current
   marginal) drops below `KL_TOL` or `MAX_ITER` iterations are reached (see the note below);
5. the resulting marginals feed the usual prior / BMU / BMU+EnKS stages and RUL statistics.

Differences with the MCMC pipeline, on purpose: the KL is computed exactly on the grid (the MCMC
version compares two 10 000-sample histograms with independent bin ranges, and stops mostly when the
estimate hits `inf`, i.e. on a sampling artefact); the returned marginals include the last posterior;
tails below `RELATIVE_TAIL` x max are trimmed from the support.

**Note on convergence.** With exact posteriors the refine loop does not settle by itself: the
HSIC-active variable keeps changing from one iteration to the next and each change moves the other
inputs' training mean, so the exact KL stays O(1). The loop is therefore capped at `MAX_ITER`
iterations (6, close to the ~5-6 iterations per cycle of the MCMC runs), and it also stops cleanly if
the calibration ensemble can no longer be fitted on a near-degenerate DoE. `converged` in the summary
table tells which cycles actually met `KL_TOL`. Outputs go to `results_grid/` in the same layout as `results/`, so `analyze_saved_results`
can read them by pointing `RESULTS_DIR` there. Runtime: a few minutes per refinement iteration
(the calibration ensemble is rebuilt each time), roughly 30-40 min for both SGs.

In [ ]:
import os
import sys

# Run from the application root so that DATA/, results*/ and figures/ resolve, and so that
# the application package is importable (pip install -e ../.. makes bayes_fusion importable).
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
os.makedirs('figures/preprocessing', exist_ok=True)
os.makedirs('figures/analysis', exist_ok=True)

In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import openturns as ot
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from clogging_prognostics import (CloggingPipeline, PipelineConfig,
                                  compute_prognostics_stages, first_crossing_rul)
from clogging_prognostics.assimilation import CycleResult
from clogging_prognostics.data_io import build_doe
from bayes_fusion.screening import select_active_index
from bayes_fusion.surrogates import build_calibration_ensemble
from bayes_fusion.mcmc import BayesCalibrationMCMC, sobol_simplex_weights

CONFIG_FILE = 'configs/rul_tables_config.json'
_cfg_json = json.load(open(CONFIG_FILE))
COMMON = _cfg_json['common']
SG_CONFIGS = {sg['sg_tag']: sg for sg in _cfg_json['steam_generators']}   # keyed by SG tag (SGA, SGB)
SG_TAGS = list(SG_CONFIGS)
SG_DISPLAY = {a: sg['display'] for a, sg in SG_CONFIGS.items()}       # SG A, SG B
THRESHOLDS = {a: sg['threshold'] for a, sg in SG_CONFIGS.items()}      # tau_c^* (%) per SG

NB_WEIGHTS = COMMON.get('nb_weights', 1024)   # fixed Sobol design size (power of two)
GRID_POINTS = 400                              # grid resolution over the active marginal's support
RELATIVE_TAIL = 1e-12                          # trim grid tails below RELATIVE_TAIL x max density
KL_TOL = 1e-4                                  # exact KL(post_k || prior_k) stopping tolerance
MAX_ITER = 6                                   # cap on refinement iterations per cycle (see intro)
N_RUL = COMMON.get('n_rul_samples', 1000)
OUT_DIR = 'results_grid'
MCMC_DIR = 'results'                           # saved MCMC run, for comparison

STAGE_ORDER = ['prior', 'bmu', 'bmu_enks']
STAGE_LABEL = {'prior': 'Prior', 'bmu': 'BMU', 'bmu_enks': 'BMU+EnKS'}
PARAM_LABEL = {'alpha': r'$\alpha$', 'beta': r'$\beta$', 'eps_e': r'$\epsilon_e$', 'eps_c': r'$\epsilon_c$',
               'd_p': r'$d_p$', 'g_p0': r'$\Gamma_{p}(0)$', 'a_v': r'$a_v$'}
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

## Exact grid posterior building blocks

`calibration_on_grid` evaluates each ensemble member on the grid (they depend only on the active
parameter); `grid_loglik` aggregates them with the fixed weight design; `trimmed_posterior`
normalizes and trims negligible tails; `grid_kl` is the exact KL between two grid densities.

In [ ]:
def calibration_on_grid(metamodels, scaler, data, tidx, xg):
    funcs = [BayesCalibrationMCMC.CalibrationFunction(m, data, tidx, scaler) for m in metamodels]
    return np.stack([np.stack([np.asarray(f([x])).reshape(-1) for f in funcs]) for x in xg])  # (G, M, T)


def grid_loglik(cal, data, tidx, weights):
    W = weights.shape[0]
    out = np.empty(cal.shape[0])
    for g in range(cal.shape[0]):
        mo = weights @ cal[g]                                   # (W, T)
        ll = np.zeros(W)
        for d, idx in zip(data, tidx):
            diff = d[:, 1][None, :] - mo[:, idx]
            if diff.shape[1] == 0:                              # channel without observations
                continue
            ll += -np.log(np.sum(diff ** 2, axis=1)) * diff.shape[1] / 2.0
        out[g] = logsumexp(ll) - np.log(W)
    return out


def trimmed_posterior(xg, L):
    p = np.exp(L - L.max())
    p /= p.sum()
    keep = np.where(p >= RELATIVE_TAIL * p.max())[0]
    xg, p = xg[keep[0]:keep[-1] + 1], p[keep[0]:keep[-1] + 1]
    return xg, p / p.sum()


def grid_kl(xg, p, prior_xg=None, prior_p=None):
    # KL(p || q) on the grid xg; q is the prior density (uniform if none) interpolated onto xg
    if prior_xg is None:
        q = np.full_like(p, 1.0 / p.size)
    else:
        dens = prior_p / np.gradient(prior_xg) if prior_xg.size > 1 else prior_p
        q = np.interp(xg, prior_xg, dens, left=0.0, right=0.0)
        q = np.maximum(q / q.sum(), 1e-300)
    m = p > 0
    return float(np.sum(p[m] * np.log(p[m] / q[m])))

## Refine loop with the exact posterior

Same steps as `assimilate_cycle`, with the MCMC replaced by the grid posterior. `history` keeps every
iteration's exact posterior (grid, probabilities, active variable, KL) for plotting.

In [ ]:
def assimilate_cycle_grid(cycle_index, pipe, X0, y0):
    cfg, dataset = pipe.cfg, pipe.dataset
    marginals = cfg.prior.marginals()
    scalers = list(pipe.per_variable_scalers)
    X, y = X0.copy(), y0.copy()
    field_times = dataset.data_without_time_division[:, 0] * 24
    data = dataset.data_with_time_division[cycle_index]
    tidx = dataset.time_indices[cycle_index]

    weights, dens_by_index, history, kl_history = None, {}, [], []
    converged = False
    for iteration in range(MAX_ITER):
        y_interp = np.asarray([np.interp(field_times, dataset.simulation_time * 24, y[i, :])
                               for i in range(len(y))])
        X_train, _, y_train, _ = train_test_split(X, y_interp, train_size=cfg.train_size, random_state=42)
        active = select_active_index(X, y)
        try:
            metamodels = build_calibration_ensemble(X_train, y_train, cfg, field_times, active)
        except TypeError as exc:      # OpenTURNS GP fit infeasible on a (nearly) degenerate DoE
            print(f'  cycle {cycle_index} stopped at iteration {iteration + 1}: '
                  f'calibration ensemble could not be fitted ({exc})', flush=True)
            break
        if weights is None:
            weights = sobol_simplex_weights(len(metamodels), NB_WEIGHTS, seed=0)

        lo = marginals[active].getRange().getLowerBound()[0]
        hi = marginals[active].getRange().getUpperBound()[0]
        xg = np.linspace(lo, hi, GRID_POINTS)
        cal = calibration_on_grid(metamodels, scalers[active], data, tidx, xg)
        xg, p = trimmed_posterior(xg, grid_loglik(cal, data, tidx, weights))

        prior_xg, prior_p = dens_by_index.get(active, (None, None))
        kl = grid_kl(xg, p, prior_xg, prior_p)
        kl_history.append(kl)
        history.append(dict(iteration=iteration, active=active, xg=xg, p=p, kl=kl))
        print(f'  cycle {cycle_index} iteration {iteration + 1}: active={cfg.prior.names[active]}, '
              f'support [{xg[0]:.3e}, {xg[-1]:.3e}], KL={kl:.3e}', flush=True)

        marginals[active] = ot.UserDefined(ot.Sample(xg.reshape(-1, 1)), [float(v) for v in p])
        dens_by_index[active] = (xg, p)
        if kl < KL_TOL:
            converged = True
            break

        prior = ot.JointDistribution(marginals)
        X_phys = np.asarray(prior.getSample(dataset.n_max))
        scalers = []
        for k in range(X_phys.shape[1]):
            s = StandardScaler()
            s.fit(X_phys[:, k].reshape(-1, 1))
            scalers.append(s)
        X = pipe.global_scaler.transform(X_phys)
        y = pipe.base_surrogate.predict(X)

    result = CycleResult(cycle_index=cycle_index, marginals=marginals, active_index=history[-1]['active'],
                         n_iterations=len(history), converged=converged, kl_history=kl_history)
    return result, history

## Run both steam generators

`CloggingPipeline.setup` loads the data and fits the base KLE-GP surrogate (used to regenerate the
DoE and to emulate the trajectories). Then every cycle is assimilated with the exact posterior, and
`compute_prognostics_stages` builds the prior / BMU / BMU+EnKS trajectory ensembles.

In [ ]:
def build_pipeline(sg):
    cfg = PipelineConfig(
        base_directory=COMMON['base_directory'],
        input_dimension=COMMON.get('input_dimension', 7),
        n_doe_samples=COMMON.get('n_doe_samples', 1000),
        n_cycles=sg['n_cycles'],
        clogging_threshold=sg['threshold'],
    )
    cfg.mcmc.nb_weights = NB_WEIGHTS
    return CloggingPipeline(cfg).setup(sg_tag=sg['sg_tag'])


runs = {}
for sg_tag in SG_TAGS:
    t0 = time.time()
    sg = SG_CONFIGS[sg_tag]
    print(f'=== {SG_DISPLAY[sg_tag]} ===', flush=True)
    pipe = build_pipeline(sg)
    ot.RandomGenerator.SetSeed(pipe.cfg.assimilation.random_seed)
    results, histories = [], []
    for c in range(pipe.cfg.n_cycles):
        X0, y0, _ = build_doe(pipe.dataset, pipe.cfg)
        res, hist = assimilate_cycle_grid(c, pipe, X0, y0)
        results.append(res)
        histories.append(hist)
    stages = compute_prognostics_stages(pipe, results, pipe.dataset, sg['split_sim_hours'],
                                        sg['prognostics_split_hours'], N_RUL)
    runs[sg_tag] = dict(pipe=pipe, results=results, histories=histories, stages=stages)
    print(f'{SG_DISPLAY[sg_tag]} done in {(time.time() - t0) / 60:.1f} min', flush=True)

## Convergence summary

In [ ]:
rows = []
for sg_tag, run in runs.items():
    names = run['pipe'].cfg.prior.names
    for res in run['results']:
        rows.append(dict(SG=SG_DISPLAY[sg_tag], cycle=res.cycle_index, active=names[res.active_index],
                         iterations=res.n_iterations, converged=res.converged,
                         final_KL=res.kl_history[-1]))
display(pd.DataFrame(rows))

## Save outputs (same layout as `results/`)

`results_grid/posteriors/<SG>_cycle<i>_posterior.csv` and `results_grid/trajectories/<SG>_<stage>[_time].npy`,
so `analyze_saved_results` can read them with `RESULTS_DIR = 'results_grid'`.

In [ ]:
os.makedirs(os.path.join(OUT_DIR, 'posteriors'), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'trajectories'), exist_ok=True)
for sg_tag, run in runs.items():
    names = list(run['pipe'].cfg.prior.names)
    for res in run['results']:
        samples = np.asarray([np.asarray(m.getSample(N_RUL)).reshape(-1) for m in res.marginals]).T
        pd.DataFrame(samples, columns=names).to_csv(
            os.path.join(OUT_DIR, 'posteriors', f'{sg_tag}_cycle{res.cycle_index}_posterior.csv'), index=False)
    st = run['stages']
    for stage, (traj, t) in {'prior': (st.prior_traj, st.pc_time), 'bmu': (st.bmu_traj, st.pc_time),
                             'bmu_enks': (st.bmu_enks_traj, st.enks_time)}.items():
        np.save(os.path.join(OUT_DIR, 'trajectories', f'{sg_tag}_{stage}.npy'), traj)
        np.save(os.path.join(OUT_DIR, 'trajectories', f'{sg_tag}_{stage}_time.npy'), np.asarray(t))
print('saved to', OUT_DIR)

## Exact posterior vs. the saved MCMC posterior

For each SG and cycle: the exact posterior of the HSIC-active variable at every refinement iteration
(light to dark) and, in colour, the KDE of the saved MCMC samples (`results/posteriors`, produced with
the previous random-weight likelihood) when available.

In [ ]:
def sci_tick(v, _pos=None):
    if v == 0:
        return r'$0$'
    e = int(np.floor(np.log10(abs(v))))
    return rf'${v / 10**e:.1f}\times10^{{{e}}}$'


from matplotlib.ticker import FuncFormatter, MaxNLocator

for sg_tag, run in runs.items():
    names = list(run['pipe'].cfg.prior.names)
    n = len(run['results'])
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.2), squeeze=False)
    for ax, res, hist in zip(axes[0], run['results'], run['histories']):
        name = names[res.active_index]
        shades = plt.cm.Greys(np.linspace(0.3, 1.0, len(hist)))
        for h, c in zip(hist, shades):
            if names[h['active']] != name:
                continue
            ax.plot(h['xg'], h['p'] / np.gradient(h['xg']), color=c, lw=1.2)
        path = os.path.join(MCMC_DIR, 'posteriors', f'{sg_tag}_cycle{res.cycle_index}_posterior.csv')
        if os.path.exists(path):
            s = pd.read_csv(path)[name].to_numpy()
            xs = np.linspace(s.min(), s.max(), 300)
            ax.plot(xs, gaussian_kde(s)(xs), color='tab:red', lw=1.5, label='saved MCMC')
        ax.set_xlabel(PARAM_LABEL.get(name, name), fontsize=14)
        ax.set_title(f'{SG_DISPLAY[sg_tag]} cycle {res.cycle_index}', fontsize=10)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.xaxis.set_major_formatter(FuncFormatter(sci_tick))
        ax.tick_params(labelsize=7)
        ax.set_ylim(0,)
        ax.set_yticks([])
        ax.legend(fontsize=7)
    axes[0, 0].set_ylabel('Density')
    fig.tight_layout()
    plt.savefig(f'figures/analysis/{sg_tag}_exact_grid_posterior.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()

## RUL statistics: exact grid posterior vs. saved MCMC run

In [ ]:
def rul_row(sg_tag, source, stage, traj, t):
    rul = first_crossing_rul(traj, t, THRESHOLDS[sg_tag])
    if rul.size == 0:
        return dict(SG=SG_DISPLAY[sg_tag], source=source, stage=STAGE_LABEL[stage], count=0)
    return dict(SG=SG_DISPLAY[sg_tag], source=source, stage=STAGE_LABEL[stage], threshold=THRESHOLDS[sg_tag],
                count=rul.size, mean=rul.mean(), std=rul.std(), q05=np.quantile(rul, .05),
                median=np.median(rul), q95=np.quantile(rul, .95))


rows = []
for sg_tag, run in runs.items():
    st = run['stages']
    arrays = {'prior': (st.prior_traj, st.pc_time), 'bmu': (st.bmu_traj, st.pc_time),
              'bmu_enks': (st.bmu_enks_traj, st.enks_time)}
    for stage in STAGE_ORDER:
        rows.append(rul_row(sg_tag, 'exact grid', stage, *arrays[stage]))
        f = os.path.join(MCMC_DIR, 'trajectories', f'{sg_tag}_{stage}.npy')
        if os.path.exists(f):
            rows.append(rul_row(sg_tag, 'saved MCMC', stage, np.load(f),
                                np.load(os.path.join(MCMC_DIR, 'trajectories', f'{sg_tag}_{stage}_time.npy'))))
rul_df = pd.DataFrame(rows).sort_values(['SG', 'stage', 'source']).reset_index(drop=True)
display(rul_df.round(1))